# 09. TOC integration — `ai_toc` → `docpipeline`

Module under test: [`src/docpipeline/parsing/pdf/toc/`](../../../../src/docpipeline/parsing/pdf/toc/)

This notebook documents the full integration of the [`ai_toc`](https://github.com/diotsiaci/ai_toc)
logic into the `docpipeline` PDF parsing architecture.

## Source → Target mapping

| `ai_toc` module | `ai_toc` function | `docpipeline` module | `docpipeline` function | TODO |
|---|---|---|---|---|
| `native.py` | `has_native_toc` | `toc/native.py` | `has_native_toc` | TOC-002 |
| `native.py` | `extract_native_toc` | `toc/native.py` | `extract_native_toc` | TOC-002 |
| `extract_toc.py` | `extract_toc_from_pdf` | `toc/native.py` | `extract_native_toc_detailed` | TOC-002 |
| `native.py` | `nest_toc` | `toc/native.py` | `nest_toc` | TOC-002 |
| `native.py` | `clean_df_for_excel` | `toc/native.py` | `_clean_df_for_excel` (private) | TOC-002 |
| `extract_toc.py` | `clean_toc_df` | `toc/native.py` | `clean_toc_df` | TOC-002 |
| `detect_links.py` | `detect_toc_with_links_df` | `toc/links.py` | `extract_toc_from_links` | TOC-004 |
| `detect_links.py` | `detect_toc_by_numbering` | `toc/links.py` | `extract_toc_by_numbering` | TOC-004 |
| `detect_textual.py` | `extract_toc_from_pdf_dotted` | `toc/textual.py` | `extract_toc_dotted` | TOC-003 |
| `detect_textual.py` | `extract_toc_multiline` | `toc/textual.py` | `extract_toc_multiline` | TOC-003 |
| `detect_textual.py` | `extract_title_candidates` | `toc/textual.py` | `extract_title_candidates` | TOC-003 |
| `detect_textual.py` | `group_numbered_titles` | `toc/textual.py` | `group_numbered_titles` | TOC-003 |
| `add_toc.py` | `add_dataframe_toc_to_pdf` | `toc/bookmarks.py` | `add_dataframe_toc_to_pdf` | TOC-002 |
| `detect_mot_cle.py` | `detect_toc_pages` | `toc/gpt.py` | `find_toc_pages` | TOC-005 |
| `detect_mot_cle.py` | `extract_toc_with_fitz` | `toc/gpt.py` | `extract_raw_toc_text` | TOC-005 |
| `detect_mot_cle.py` | `extract_structured_toc_with_chatgpt` | `toc/gpt.py` | `_extract_structured_with_openai` | TOC-005 |
| `detect_mot_cle.py` | `extract_toc_with_gpt` | `toc/gpt.py` | `extract_toc_with_gpt` | TOC-005 |

**Strategy cascade (zero LLM first, LLM as last resort):**
```
native bookmarks → internal links → dotted/multiline text → GPT (fallback)
```

In [1]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src').exists():
            return candidate
    raise RuntimeError('Could not find repository root')

REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC = REPO_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

REPO_ROOT

PosixPath('/Users/sylvereakambi/Documents/Project/FASIA/toc_detection/EXTERNAL/doc_intelligence')

In [2]:
import os
import pandas as pd
from IPython.display import Markdown, display

from docpipeline.parsing.pdf.toc import (
    # TOC-001 — heuristic detection
    detect_toc,
    score_page,
    # TOC-002 — native bookmarks  (ai_toc: native.py + extract_toc.py)
    has_native_toc,
    extract_native_toc,
    extract_native_toc_detailed,
    clean_toc_df,
    nest_toc,
    export_toc_to_json,
    export_toc_to_excel,
    # TOC-003 — textual extraction  (ai_toc: detect_textual.py)
    extract_toc_dotted,
    extract_toc_multiline,
    extract_title_candidates,
    group_numbered_titles,
    # TOC-004 — link extraction  (ai_toc: detect_links.py)
    extract_toc_from_links,
    extract_toc_by_numbering,
    # TOC-005 — LLM extraction  (ai_toc: detect_mot_cle.py)
    extract_toc_with_gpt,
    find_toc_pages,
    extract_raw_toc_text,
    # Bookmark writing  (ai_toc: add_toc.py)
    add_dataframe_toc_to_pdf,
)

pd.set_option("display.max_colwidth", 120)

In [3]:
candidate_pdfs = [
    REPO_ROOT / 'data' / 'CG contrats MRH' / 'CG MRH_GMF.pdf',
    REPO_ROOT / 'data' / 'reports' / 'sdg_7_2025.pdf',
    REPO_ROOT / 'data' / 'insurance' / 'StateFarm.pdf',
]

PDF = next((path for path in candidate_pdfs if path.exists()), None)
if PDF is None:
    raise FileNotFoundError('No demo PDF found in the local data folder')

PDF

PosixPath('/Users/sylvereakambi/Documents/Project/FASIA/toc_detection/EXTERNAL/doc_intelligence/data/CG contrats MRH/CG MRH_GMF.pdf')

## 1. Heuristic detection

In [4]:
detection = detect_toc(PDF, max_pages=5, threshold=0.5)
display(pd.DataFrame([detection.to_dict()]))

,has_toc,confidence,toc_pages,reason
0,True,0.5,"[4, 5]",Multiple lines ending with page numbers (12 occurrences); Hierarchical numbering structure detected (27 entries); Hi...


## 2. Native PDF bookmarks

In [5]:
display(Markdown(f'Native bookmarks present: `{has_native_toc(PDF)}`'))

native_df = extract_native_toc(PDF)
display(native_df.head(20) if not native_df.empty else pd.DataFrame({'result': ['no native bookmarks']}))

detailed_native_df = extract_native_toc_detailed(PDF)
display(detailed_native_df.head(20) if not detailed_native_df.empty else pd.DataFrame({'result': ['no detailed native bookmarks']}))

Native bookmarks present: `True`

,level,title,page,indicator
0,1,1ère de couverture,1,L1
1,1,Sommaire,4,L2
2,1,1 Les dispositions générales,7,L3
3,2,1.1 COMMENT EST RÉGI VOTRE CONTRAT ?,8,L3.1
4,2,1.2 OÙ S’APPLIQUE VOTRE CONTRAT ?,8,L3.2
5,2,1.3 LES DÉFINITIONS ET CE QU’IL EST IMPORTANT DE SAVOIR POUR L’APPLICATION DE VOTRE CONTRAT,9,L3.3
6,2,1.4 CE QUI N’EST PAS ASSURÉ PAR VOTRE CONTRAT,20,L3.4
7,1,2 Les garanties personnelles,23,L4
8,2,2.1 LA GARANTIE RESPONSABILITÉ CIVILE PERSONNELLE OU FAMILIALE,24,L4.1
9,2,2.2 LA GARANTIE DÉFENSE PÉNALE ET RECOURS SUITE À ACCIDENT,27,L4.2


,level,title,displayed_page,page,kind,xref,zoom,collapse,to_x,to_y
0,1,1ère de couverture,1,0,1,774,0.0,None,-151.0,0.276001
1,1,Sommaire,4,3,1,805,0.0,None,-151.0,-155.724000
2,1,1 Les dispositions générales,7,6,1,801,0.0,False,-151.0,4.276001
3,2,1.1 COMMENT EST RÉGI VOTRE CONTRAT ?,8,7,1,803,0.0,None,-151.0,-210.724000
4,2,1.2 OÙ S’APPLIQUE VOTRE CONTRAT ?,8,7,1,810,0.0,None,-151.0,-210.724000
5,2,1.3 LES DÉFINITIONS ET CE QU’IL EST IMPORTANT DE SAVOIR POUR L’APPLICATION DE VOTRE CONTRAT,9,8,1,808,0.0,None,-151.0,-180.724000
6,2,1.4 CE QUI N’EST PAS ASSURÉ PAR VOTRE CONTRAT,20,19,1,804,0.0,None,-151.0,8.276001
7,1,2 Les garanties personnelles,23,22,1,797,0.0,False,-151.0,63.276000
8,2,2.1 LA GARANTIE RESPONSABILITÉ CIVILE PERSONNELLE OU FAMILIALE,24,23,1,799,0.0,None,-151.0,-117.724000
9,2,2.2 LA GARANTIE DÉFENSE PÉNALE ET RECOURS SUITE À ACCIDENT,27,26,1,818,0.0,None,-151.0,-27.723999


## 3. Link-based and numbering extraction

In [6]:
links_df = extract_toc_from_links(PDF, max_pages=10)
numbering_df = extract_toc_by_numbering(PDF, max_pages=5)

display(Markdown('### Internal links'))
display(links_df.head(20) if not links_df.empty else pd.DataFrame({'result': ['no internal links']}))

display(Markdown('### Numbered lines'))
display(numbering_df.head(20) if not numbering_df.empty else pd.DataFrame({'result': ['no numbered TOC lines']}))

### Internal links

,result
0,no internal links


### Numbered lines

,text,page_num
0,1,1
1,1,2
2,1,3
3,9 à,20
4,1,4
5,20/,21
6,2,1
7,24 à,27
8,2,2
9,2,3


## 4. Textual extraction

In [7]:
dotted_df = extract_toc_dotted(PDF)
multiline_df = extract_toc_multiline(PDF)

display(Markdown('### Dotted leaders'))
display(dotted_df.head(20) if not dotted_df.empty else pd.DataFrame({'result': ['no dotted leaders']}))

display(Markdown('### Multiline leaders'))
display(multiline_df.head(20) if not multiline_df.empty else pd.DataFrame({'result': ['no multiline leaders']}))

grouped_df = group_numbered_titles(multiline_df) if not multiline_df.empty else multiline_df
display(Markdown('### Grouped numbered titles'))
display(grouped_df.head(20) if not grouped_df.empty else pd.DataFrame({'result': ['no grouped entries']}))

### Dotted leaders

,result
0,no dotted leaders


### Multiline leaders

,text,page_num,source_page
0,"diation de l’Assurance, TSA 50110",75441,102


### Grouped numbered titles

,text,page_num,source_page
0,"diation de l’Assurance, TSA 50110",75441,102


## 5. Choose the best extracted TOC DataFrame

In [8]:
candidates = [
    ('native', native_df.rename(columns={'title': 'text', 'page': 'page_num'})),
    ('links', links_df),
    ('numbering', numbering_df),
    ('dotted', dotted_df),
    ('multiline', grouped_df),
]

source_name, best_toc_df = next(
    ((name, df) for name, df in candidates if {'text', 'page_num'}.issubset(df.columns) and not df.empty),
    ('none', pd.DataFrame(columns=['text', 'page_num'])),
)

display(Markdown(f'Chosen source: `{source_name}`'))
display(best_toc_df[['text', 'page_num']].head(30) if not best_toc_df.empty else best_toc_df)

Chosen source: `native`

,text,page_num
0,1ère de couverture,1
1,Sommaire,4
2,1 Les dispositions générales,7
3,1.1 COMMENT EST RÉGI VOTRE CONTRAT ?,8
4,1.2 OÙ S’APPLIQUE VOTRE CONTRAT ?,8
5,1.3 LES DÉFINITIONS ET CE QU’IL EST IMPORTANT DE SAVOIR POUR L’APPLICATION DE VOTRE CONTRAT,9
6,1.4 CE QUI N’EST PAS ASSURÉ PAR VOTRE CONTRAT,20
7,2 Les garanties personnelles,23
8,2.1 LA GARANTIE RESPONSABILITÉ CIVILE PERSONNELLE OU FAMILIALE,24
9,2.2 LA GARANTIE DÉFENSE PÉNALE ET RECOURS SUITE À ACCIDENT,27


## 6. Optional bookmark export

In [9]:
OUTPUT_DIR = REPO_ROOT / 'notebooks' / '_outputs' / 'pdf_toc'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PDF = OUTPUT_DIR / f'{PDF.stem}_with_detected_toc.pdf'

if best_toc_df.empty:
    display(pd.DataFrame({'result': ['no TOC entries available for bookmark export']}))
else:
    written = add_dataframe_toc_to_pdf(best_toc_df[['text', 'page_num']], PDF, OUTPUT_PDF)
    display(pd.DataFrame([{'output_pdf': str(OUTPUT_PDF), 'bookmarks_written': written}]))

,output_pdf,bookmarks_written
0,/Users/sylvereakambi/Documents/Project/FASIA/toc_detection/EXTERNAL/doc_intelligence/notebooks/_outputs/pdf_toc/CG M...,65


## 7. LLM extraction — `ai_toc.detect_mot_cle` → `docpipeline.parsing.pdf.toc.gpt`

**TODO-TOC-005** — Last-resort strategy when all heuristic methods return empty results.

Requires `pip install 'docpipeline[llm]'` and an `OPENAI_API_KEY` environment variable.

Differences vs `ai_toc`:
- Uses `client.beta.chat.completions.parse` (openai ≥1.12) instead of `client.responses.parse` (openai v2 only)
- Default model is `gpt-4o-mini` (economical) vs `gpt-4.1` in ai_toc
- Returns an empty DataFrame (never None) on failure, consistent with other extractors

In [10]:
import fitz  # noqa: F401 — used by find_toc_pages / extract_raw_toc_text helpers
from docpipeline.parsing.pdf.toc.reader import extract_text_from_first_pages

# Step 7a — preview the text that will be sent to the LLM
# Same logic as the base toc_detector project: read the first N pages, no keyword filter
pages = extract_text_from_first_pages(PDF, max_pages=5)
raw_text_for_llm = "\n\n--- page break ---\n\n".join(
    p["text"] for p in pages if p["text"].strip()
)

display(Markdown(f"Pages read: `{[p['page_number'] for p in pages]}`"))
display(Markdown(f"Characters sent to the LLM: `{len(raw_text_for_llm)}`"))
print(raw_text_for_llm[:800])  # preview — truncate for readability

# Also show which pages contain TOC keywords (for info — not used in extraction)
with fitz.open(str(PDF)) as _doc:
    toc_keyword_pages = find_toc_pages(_doc)
display(Markdown(f"Pages with TOC keywords (info only): `{toc_keyword_pages}`"))

Pages read: `[1, 2, 3, 4, 5]`

Characters sent to the LLM: `2975`

HABITATION
Conditions Générales
Novembre 2024

--- page break ---

Vous venez de souscrire un contrat
pour votre habitation,
nous vous remercions de votre confiance.
N’hésitez pas à consulter
votre Conseiller GMF
pour toute information complémentaire.
Les entreprises d’assurances agréées en France sont placées sous le contrôle
de l’Autorité de Contrôle Prudentiel et de Résolution (A.C.P.R.) :
4, place de Budapest - CS 92459 - 75436 Paris cedex 09.

--- page break ---

S
ommaire
1 Les dispositions généraLes
1.1 C omment est régi votre contrat ? 8
1.2 O ù s’applique votre contrat ? 8
1.3 Les définitions et ce qu’il est important de savoir
pour l’application de votre contrat 9 à 20
1.4 C e qui n’est pas assuré par votre contrat 20/21
2 Les garanties personneLLes
2.1 L a Garantie Responsabilit


Pages with TOC keywords (info only): `[4, 5, 6]`

In [11]:
# Step 7b — run GPT extraction (requires OPENAI_API_KEY)
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

if OPENAI_API_KEY:
    gpt_df = extract_toc_with_gpt(PDF, api_key=OPENAI_API_KEY, model="gpt-4o-mini")
    display(Markdown(f"GPT TOC entries found: `{len(gpt_df)}`"))
    display(gpt_df if not gpt_df.empty else pd.DataFrame({"result": ["LLM returned no entries"]}))
else:
    display(Markdown(
        "⚠️ `OPENAI_API_KEY` not set — skipping live LLM call.  "
        "Set the variable and re-run to see structured output from `extract_toc_with_gpt`."
    ))
    display(pd.DataFrame({
        "column": ["level", "title", "page_num"],
        "description": [
            "Hierarchical depth (1=top, 2=subsection…)",
            "Section title as extracted by the LLM",
            "Target page number",
        ],
    }))

⚠️ `OPENAI_API_KEY` not set — skipping live LLM call.  Set the variable and re-run to see structured output from `extract_toc_with_gpt`.

,column,description
0,level,"Hierarchical depth (1=top, 2=subsection…)"
1,title,Section title as extracted by the LLM
2,page_num,Target page number


## 8. Full strategy cascade — zero LLM → LLM fallback

Complete decision logic integrating all ai_toc strategies in priority order.
This is the recommended production usage when `OPENAI_API_KEY` is available.

In [12]:
def extract_best_toc(
    pdf_path,
    openai_api_key: str | None = None,
    gpt_model: str = "gpt-4o-mini",
) -> tuple[str, pd.DataFrame]:
    """
    Full strategy cascade (mirrors ai_toc recommendation logic).

    Priority : native → links → textual (dotted/multiline) → GPT (if key provided)
    Returns  : (strategy_name, toc_df with columns text + page_num [+ level])
    """
    # 1. Native bookmarks  (TOC-002 — ai_toc: native.py)
    if has_native_toc(pdf_path):
        df = extract_native_toc(pdf_path).rename(columns={"title": "text", "page": "page_num"})
        return "native", df

    # 2. Internal hyperlinks  (TOC-004 — ai_toc: detect_links.py)
    links_df = extract_toc_from_links(pdf_path, max_pages=10)
    if not links_df.empty:
        return "links", links_df

    # 3a. Dotted leader lines  (TOC-003 — ai_toc: detect_textual.py)
    dotted_df = extract_toc_dotted(pdf_path)
    if not dotted_df.empty:
        return "textual_dotted", dotted_df

    # 3b. Multiline leaders  (TOC-003 — ai_toc: detect_textual.py)
    multiline_df = extract_toc_multiline(pdf_path)
    grouped_df = group_numbered_titles(multiline_df) if not multiline_df.empty else multiline_df
    if not grouped_df.empty:
        return "textual_multiline", grouped_df

    # 4. LLM fallback  (TOC-005 — ai_toc: detect_mot_cle.py)
    if openai_api_key:
        gpt_df = extract_toc_with_gpt(pdf_path, api_key=openai_api_key, model=gpt_model)
        if not gpt_df.empty:
            return "gpt", gpt_df.rename(columns={"title": "text"})

    return "none", pd.DataFrame(columns=["text", "page_num"])


strategy, final_toc_df = extract_best_toc(PDF, openai_api_key=os.environ.get("OPENAI_API_KEY"))

display(Markdown(f"### Strategy selected: `{strategy}`  ({len(final_toc_df)} entries)"))
display(final_toc_df[["text", "page_num"]].head(30) if not final_toc_df.empty else final_toc_df)

### Strategy selected: `native`  (65 entries)

,text,page_num
0,1ère de couverture,1
1,Sommaire,4
2,1 Les dispositions générales,7
3,1.1 COMMENT EST RÉGI VOTRE CONTRAT ?,8
4,1.2 OÙ S’APPLIQUE VOTRE CONTRAT ?,8
5,1.3 LES DÉFINITIONS ET CE QU’IL EST IMPORTANT DE SAVOIR POUR L’APPLICATION DE VOTRE CONTRAT,9
6,1.4 CE QUI N’EST PAS ASSURÉ PAR VOTRE CONTRAT,20
7,2 Les garanties personnelles,23
8,2.1 LA GARANTIE RESPONSABILITÉ CIVILE PERSONNELLE OU FAMILIALE,24
9,2.2 LA GARANTIE DÉFENSE PÉNALE ET RECOURS SUITE À ACCIDENT,27
